# SmartFlow — 03 · Experiment comparison & final report

This notebook **reads** the artefacts written by notebooks 01 and 02: data gate, validation reports, `metrics.json`, `epochs.csv`, checkpoints and summary CSVs. From them it builds the comparison table, plots, per-class analysis and a Markdown report.

It **measures** only these three things, each with the same protocol for every experiment, and caches the results on Drive:
1. **Pretrained baseline (zero-shot).** The COCO-pretrained `yolov8n.pt`, before any fine-tuning, is evaluated on each experiment's **test** split. Its classification head is restricted to the COCO classes that correspond to the dataset's classes.
2. **Inference speed, re-measured on one device.** All fine-tuned checkpoints are timed on the current runtime with the same protocol, because notebook 02 timed E1 on a T4 GPU but E2 on CPU.
3. **Confusion matrices at confidence 0.25**, for visual inspection only. Notebook 02's matrices were produced by the mAP evaluation at confidence 0.001, where the background column is dominated by very low-confidence boxes.

**What it never does:**
- invent a number;
- train;
- use a test result to select or tune anything.

Experiments that were skipped or failed appear with their reason and **empty** metric cells.

Run with *Runtime → Run all*. A GPU makes the measurements faster but is not required. Each measurement is cached, so reruns only regenerate tables, plots and the report.

## 1 · Configuration
`DATASET_NOTES` holds **documented** facts from `docs/dataset_audit.md` and the official dataset pages, labelled as such. Everything else in the report is either **measured** in this notebook or **read** from the run artefacts.

In [ ]:
# 1. Configuration
from pathlib import Path

CFG = {
    "drive_mount": "/content/drive",
    "drive_root": "/content/drive/MyDrive/SmartFlow",
    "local_root": "/content/smartflow_work",
    "ultralytics_version": "8.4.160",
    "install_dependencies": True,
    "seed": 42,
    "run_baseline": True,              # zero-shot COCO-pretrained yolov8n on each test split
    "remeasure_speed": True,           # same-device latency for every fine-tuned checkpoint
    "confusion_matrix_conf": 0.25,     # visual-only confusion matrices (None = skip)
    "measure_dataset_stats": True,     # resolution / density / box-size statistics from the label files
    "eval": {"imgsz": 640, "conf": 0.001, "iou": 0.7, "batch": 16},   # identical to notebook 02
    "speed": {"images": 200, "warmup": 10, "imgsz": 640},           # identical to notebook 02
    # Poor-class rule (stated in the report): test mAP@50 below this OR recall below this.
    "poor_class_map50": 0.50,
    "poor_class_recall": 0.50,
    # Pretrained baseline: which COCO classes stand in for each dataset class (COCO ids: 2 car, 3 motorcycle,
    # 5 bus, 7 truck). UA-DETRAC's single 'vehicle' class covers cars, buses, vans and other vehicles but no
    # motorcycles, so the baseline merges COCO car+bus+truck into one class.
    "baseline_mapping": {
        "ua_detrac_mirror_1cls": {"coco_ids": [2, 5, 7], "coco_names": ["car", "bus", "truck"], "single_cls": True},
        "bdd100k_4cls": {"coco_ids": [2, 5, 7, 3], "coco_names": ["car", "bus", "truck", "motorcycle"],
                         "single_cls": False},
    },
}

# Results REPORTED by Raza et al. (2025), quoted verbatim from the paper's tables (not reproduced, not measured here).
# Their dataset is private (Karachi, 8 classes); the paper does not state the IoU threshold of its "mAP".
PAPER = {
    "citation": "M. Raza et al., IEEE Access, vol. 13, pp. 153586-153613, 2025, doi:10.1109/ACCESS.2025.3602844",
    "dataset": "private Karachi dataset (Allama Shabbir Usmani road), 8 classes, 1,373 source images; "
               "2,726 after augmentation, 80/10/10 split (Table 4, Sec. III-A)",
    "training": "YOLOv8n: AdamW, lr 0.001, batch 16, 100 epochs (Table 7)",
    "table12": {  # Tesla T4 workstation (Sec. IV); colour scale = input variant
        "hardware": "NVIDIA Tesla T4 (GPU workstation, per Sec. IV)",
        "models": {"YOLOv7-tiny": {"size_mb": 11.8, "mAP": {"RGB": 86.40, "Gray": 83.70, "Green": 80.00},
                                   "fps": {"RGB": 37.39, "Gray": 36.83, "Green": 33.95}},
                   "YOLOv8-nano": {"size_mb": 6.0, "mAP": {"RGB": 89.30, "Gray": 90.20, "Green": 94.90},
                                   "fps": {"RGB": 74.67, "Gray": 73.56, "Green": 75.36}},
                   "YOLOv8-small": {"size_mb": 21.5, "mAP": {"RGB": 90.50, "Gray": 91.00, "Green": 95.00},
                                    "fps": {"RGB": 69.59, "Gray": 69.10, "Green": 70.07}},
                   "YOLO-NAS-small": {"size_mb": 244.4, "mAP": {"RGB": 93.90, "Gray": 95.40, "Green": 87.90},
                                      "fps": {"RGB": 4.26, "Gray": 3.70, "Green": 4.81}}},
    },
    "table13": {  # cross-domain mAP, unaugmented vs augmented training (Table 14 in the paper duplicates it)
        "YOLOv7-tiny": {"KITTI": (67.67, 83.30), "PASCAL VOC": (80.10, 86.50),
                        "Allama Shabbir Usmani": (81.00, 86.40), "Teen Talwar": (62.00, 84.35)},
        "YOLOv8-nano": {"KITTI": (72.20, 89.00), "PASCAL VOC": (82.50, 95.00),
                        "Allama Shabbir Usmani": (83.50, 89.30), "Teen Talwar": (75.00, 88.50)},
    },
    "table15": {  # edge devices, TensorRT (Sec. IV-B): (mAP %, Jetson Nano FPS, Jetson Xavier FPS); None = '-'
        "Allama Shabbir Usmani": {"YOLOv7-tiny": (82.0, 23, 27), "YOLOv8-nano": (83.7, None, 35)},
        "Teen Talwar": {"YOLOv7-tiny": (78.4, 19, 24), "YOLOv8-nano": (81.45, None, 33.5)},
    },
}
DATASET_NAMES = {"qmul_junction": "QMUL Junction", "aic2021": "AI City Challenge 2021"}   # not evaluated
DATASET_NOTES = {   # documented (not measured) - source: docs/dataset_audit.md and official dataset pages
    "ua_detrac_mirror": {
        "name": "UA-DETRAC",
        "source": "Hugging Face mirror blues111/ua_detrac (third-party repackaging of UA-DETRAC)",
        "domain": "fixed, elevated traffic-surveillance cameras at urban roads in Beijing and Tianjin; video frames",
        "annotation": "boxes from the original UA-DETRAC annotation (per-frame tracking annotation); the mirror "
                      "keeps only 'vehicle' + 'ignored' regions (vehicle types car/bus/van/others are merged)",
        "class_definition": "1 class: vehicle (car, bus, van, others merged; motorcycles/bicycles not annotated)",
        "preparation": "ignored regions masked grey; sequence-level splits; every 10th frame",
    },
    "bdd100k": {
        "name": "BDD100K",
        "source": "UC Berkeley BDD100K server (100K images + official 2018 detection labels)",
        "domain": "dash-cam images from US cities, diverse weather and time of day (day/night)",
        "annotation": "official 2018 detection labels (the later 2020 revision with extra quality checks "
                      "could not be downloaded)",
        "class_definition": "4 classes mapped from BDD categories: car, bus, truck (+trailer), motorcycle; "
                            "pedestrian/rider/bicycle/traffic light/sign dropped; train and 'other vehicle' masked",
        "preparation": "test = official val; val = 10% of official train; one frame per video",
    },
}
print("Configuration loaded.")

## 2 · Dependencies, Drive, environment
This uses the same pinned `ultralytics` as notebook 02. `openpyxl` is needed only for the Excel export, and the notebook continues without it.

In [ ]:
# 2. Dependencies + Drive + environment
import importlib.metadata as md_, json, os, platform, shutil, subprocess, sys
from datetime import datetime, timezone

def _ver(p):
    try:
        return md_.version(p)
    except md_.PackageNotFoundError:
        return None

if CFG["install_dependencies"] and _ver("ultralytics") != CFG["ultralytics_version"]:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={CFG['ultralytics_version']}"],
                       capture_output=True, text=True)
    assert r.returncode == 0, r.stderr[-1500:]
assert _ver("ultralytics") == CFG["ultralytics_version"], f"ultralytics {_ver('ultralytics')} != pinned version"
os.environ.setdefault("YOLO_VERBOSE", "False")   # keep notebook output small

try:
    from google.colab import drive as _gdrive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB and not Path(CFG["drive_mount"], "MyDrive").is_dir():
    _gdrive.mount(CFG["drive_mount"])

import numpy as np
import pandas as pd
import torch
import ultralytics

DRIVE, LOCAL = Path(CFG["drive_root"]), Path(CFG["local_root"])
DIRS = {k: DRIVE / k for k in ("prepared_cache", "checkpoints", "logs", "metrics", "plots", "reports")}
DIRS.update(local_yolo=LOCAL / "yolo", local_tmp=LOCAL / "tmp", final=DRIVE / "reports" / "final_report",
            cache=DRIVE / "metrics" / "nb03_cache")
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
ENV3 = {"utc": datetime.now(timezone.utc).isoformat(), "python": sys.version.split()[0],
        "torch": torch.__version__, "ultralytics": ultralytics.__version__,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "device": str(DEVICE)}
print(ENV3)
if not torch.cuda.is_available():
    print("NOTE: no GPU - baseline evaluation, speed and confusion matrices run on CPU (slower, ~1 h in total for "
          "UA-DETRAC + BDD100K test splits). Results are cached; speed is labelled with the device it ran on.")

## 3 · Helpers
These are reading and writing utilities, plus dataset restore with a verified sha256 (the same logic as notebook 02).

In [ ]:
# 3. Helpers
import hashlib, math, random, tarfile, csv
from collections import Counter
import yaml
from PIL import Image

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}


def log(msg):
    line = f"[{datetime.now().strftime('%H:%M:%S')}] {msg}"
    print(line)
    with open(DIRS["logs"] / "nb03_report.log", "a") as f:
        f.write(line + "\n")


def to_py(o):
    if isinstance(o, dict):
        return {str(k): to_py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_py(v) for v in o]
    if isinstance(o, Path):
        return str(o)
    if hasattr(o, "item") and not isinstance(o, (str, bytes)):
        try:
            o = o.item()
        except (ValueError, AttributeError):
            return str(o)
    if isinstance(o, float) and not math.isfinite(o):
        return None
    return o


def write_json(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + ".tmp")
    tmp.write_text(json.dumps(to_py(obj), indent=2))
    tmp.replace(path)


def read_json(path):
    p = Path(path)
    return json.loads(p.read_text()) if p.exists() else None


def file_hash(path, algo="sha256", chunk=16 << 20):
    path = Path(path)
    st = path.stat()
    side = path.with_name(path.name + f".{algo}.json")
    if side.exists():
        try:
            c = json.loads(side.read_text())
            if c["size"] == st.st_size and c["mtime_ns"] == st.st_mtime_ns:
                return c["hash"]
        except (ValueError, KeyError):
            pass
    h = hashlib.new(algo)
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    try:
        side.write_text(json.dumps({"size": st.st_size, "mtime_ns": st.st_mtime_ns, "hash": h.hexdigest()}))
    except OSError:
        pass
    return h.hexdigest()


def stable_hash(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:12]


def split_files(root, split):
    d = Path(root) / "images" / split
    return sorted(p for p in d.iterdir() if p.suffix.lower() in IMG_EXTS) if d.is_dir() else []


def restore_dataset(variant, fingerprint):
    root = DIRS["local_yolo"] / variant
    marker = root / ".prepared.json"
    if marker.exists() and json.loads(marker.read_text()).get("fingerprint") == fingerprint:
        return root
    tar_path, man_path = DIRS["prepared_cache"] / f"{variant}.tar", DIRS["prepared_cache"] / f"{variant}.json"
    if not (tar_path.exists() and man_path.exists()):
        raise FileNotFoundError(f"{variant}: no local copy and no Drive cache ({tar_path}).")
    man = json.loads(man_path.read_text())
    if man["fingerprint"] != fingerprint or file_hash(tar_path) != man["sha256"]:
        raise IOError(f"{variant}: Drive cache does not match the data gate (fingerprint/sha256).")
    log(f"{variant}: restoring from Drive cache ...")
    tmp = DIRS["local_tmp"] / f"restore_{variant}"
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir(parents=True)
    with tarfile.open(tar_path) as tf:
        for m in tf.getmembers():
            t = (tmp / m.name).resolve()
            if tmp.resolve() not in t.parents or m.issym() or m.islnk():
                raise IOError(f"unsafe tar member {m.name}")
        try:
            tf.extractall(tmp, filter="data")
        except TypeError:
            tf.extractall(tmp)
    shutil.rmtree(root, ignore_errors=True)
    (tmp / variant).replace(root)
    shutil.rmtree(tmp, ignore_errors=True)
    return root


def eval_yaml(root, names, tag):
    p = DIRS["local_yolo"] / f"nb03_{tag}__eval.yaml"
    p.write_text(yaml.safe_dump({"path": str(root), "train": "images/train", "val": "images/val",
                                 "test": "images/test", "names": {i: n for i, n in enumerate(names)}}, sort_keys=False))
    return p


def f1(p, r):
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0


def fmt(v, nd=3):
    if v is None or (isinstance(v, float) and not math.isfinite(v)):
        return "n/a"
    return f"{v:.{nd}f}" if isinstance(v, float) else str(v)


def md_table(rows, cols, headers=None):
    headers = headers or cols
    out = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(cols)]
    for r in rows:
        out.append("| " + " | ".join(fmt(r.get(c)) for c in cols) + " |")
    return "\n".join(out)


print("Helpers loaded.")

## 4 · Load all experiment artefacts 🔍
This reads the summary written by notebook 02, then for every `EVALUATED` experiment its `metrics.json`, `epochs.csv` and `run_status.json`. It cross-checks that the numbers in the summary CSV equal the ones in `metrics.json`, and that the `best.pt` file exists.

**Inspect:** the status table. Experiments that are SKIPPED or not evaluated carry a reason and will have empty metrics.

In [ ]:
# 4. Load artefacts
GATE = read_json(DIRS["reports"] / "data_gate.json")
assert GATE and GATE.get("passed"), "data_gate.json missing or not passed - run notebook 01."
summary_path = DIRS["metrics"] / "summary_all_experiments.csv"
assert summary_path.exists(), f"{summary_path} not found - run notebook 02 first."
SUMMARY = pd.read_csv(summary_path)
PREFLIGHT = read_json(DIRS["reports"] / "nb02_preflight.json") or {}
PRETRAINED_SHA = PREFLIGHT.get("info", {}).get("pretrained_sha256")

EXPS, problems = [], []
for row in SUMMARY.to_dict("records"):
    e = {"id": row["experiment"], "dataset": row["dataset"], "status": row["status"],
         "variant": row["variant"] if isinstance(row.get("variant"), str) else None,
         "reason": row["reason"] if isinstance(row.get("reason"), str) else ""}
    if e["status"] == "EVALUATED":
        run_dir = Path(row["best_checkpoint"]).parent.parent
        rec = read_json(run_dir / "report" / "metrics.json")
        if rec is None:
            problems.append(f"{e['id']}: metrics.json missing in {run_dir / 'report'}")
            e.update(status="REPORT_MISSING", reason="metrics.json not found")
        else:
            for k_csv, v_json in (("test_mAP50", rec["test"]["overall"]["mAP50"]),
                                  ("test_mAP50_95", rec["test"]["overall"]["mAP50_95"]),
                                  ("test_precision", rec["test"]["overall"]["precision"]),
                                  ("test_recall", rec["test"]["overall"]["recall"])):
                if abs(float(row[k_csv]) - float(v_json)) > 1e-9:
                    problems.append(f"{e['id']}: {k_csv} differs between summary CSV and metrics.json")
            best = run_dir / "weights" / "best.pt"
            if not best.exists():
                problems.append(f"{e['id']}: best.pt missing ({best})")
            e.update(run_dir=run_dir, rec=rec, best=best, status_json=read_json(run_dir / "run_status.json") or {},
                     epochs=pd.read_csv(run_dir / "report" / "epochs.csv"),
                     gate=GATE["datasets"].get(e["dataset"], {}))
    EXPS.append(e)

print(pd.DataFrame([{"experiment": e["id"], "dataset": e["dataset"], "status": e["status"],
                     "reason": (e["reason"] or "")[:110]} for e in EXPS]).to_string(index=False))
assert not problems, "Inconsistent artefacts: " + "; ".join(problems)
EVAL = [e for e in EXPS if e["status"] == "EVALUATED"]
# notebook 01's recorded configuration (for dataset versions)
_nb01 = sorted(p for p in DIRS["logs"].glob("environment_*.json"))
NB01_CFG = json.loads(_nb01[-1].read_text()).get("config", {}) if _nb01 else {}


def dataset_version(e):
    if e["dataset"] == "ua_detrac_mirror":
        c = NB01_CFG.get("ua_detrac_mirror", {})
        return f"HF {c.get('repo', 'blues111/ua_detrac')} @ {str(c.get('revision', 'unrecorded'))[:12]}"
    if e["dataset"] == "bdd100k":
        return f"BDD100K 100K images + {e['rec'].get('label_version') or 'unrecorded'} detection labels"
    return "unrecorded"
assert EVAL, "No evaluated experiment found."
print(f"\nLoaded {len(EVAL)} evaluated experiment(s); summary CSV and metrics.json agree.")

## 5 · Held-out test verification 🔍
For each evaluated experiment this confirms, **from saved artefacts**, that the reported test metrics come from data that was not used for training or model selection:

| Evidence | Source |
|---|---|
| No file-name overlap and no byte-identical images across train, val and test | notebook 01 validation report |
| The file lists used for training equal the validated gate manifest | notebook 02 pre-training report |
| The training subset (if any) lies strictly inside the train split | notebook 02 subset report |
| The training config pointed at a train-only YAML (no `test` key) | `requested_args.data` and the run's `args.yaml` |
| Checkpoint selection used the validation metric | `metrics.json` |

Any piece of evidence that can't be found is reported as **UNVERIFIED**, never as passed.

In [ ]:
# 5. Held-out verification
def report_checks(name):
    r = read_json(DIRS["reports"] / f"{name}.json")
    return None if r is None else {c["check"]: c["ok"] for c in r["checks"]}

VERIFY = []
for e in EVAL:
    v, rec = e["variant"], e["rec"]
    rows = []
    c1 = report_checks(f"{v}_validation")
    ok1 = None if c1 is None else all(ok for k, ok in c1.items() if "overlap" in k or "byte-identical" in k)
    rows.append(("no overlap / duplicates across splits (nb01)", ok1))
    c2 = report_checks(f"{e['id']}_{v}_pretrain_checks")
    ok2 = None if c2 is None else all(ok for k, ok in c2.items() if "gate manifest" in k)
    rows.append(("file lists equal the validated gate manifest (nb02)", ok2))
    if rec.get("train_subset"):
        c3 = report_checks(f"{e['id']}_{v}_train_subset_pretrain_checks")
        ok3 = None if c3 is None else all(ok for k, ok in c3.items() if "strict subset" in k)
        rows.append((f"training subset ({rec['train_subset']['n']} imgs) inside train split", ok3))
    data_arg = str(rec["training_config"]["requested_args"]["data"])
    args_yaml = e["run_dir"] / "args.yaml"
    run_data = yaml.safe_load(args_yaml.read_text()).get("data") if args_yaml.exists() else None
    rows.append(("training used the train-only YAML (no test key)",
                 data_arg.endswith("__train.yaml") and (run_data is None or str(run_data).endswith("__train.yaml"))))
    rows.append(("checkpoint selected on validation metric", "validation" in rec["training"]["selection_metric"]))
    for check, ok in rows:
        VERIFY.append({"experiment": e["id"], "evidence": check,
                       "result": "PASS" if ok else ("UNVERIFIED" if ok is None else "FAIL")})
VERIFY_DF = pd.DataFrame(VERIFY)
print(VERIFY_DF.to_string(index=False))
HELD_OUT_OK = {e["id"]: all(r["result"] == "PASS" for r in VERIFY if r["experiment"] == e["id"]) for e in EVAL}
assert not (VERIFY_DF["result"] == "FAIL").any(), "A held-out check FAILED - do not report these test results."
print("\nHeld-out status:", HELD_OUT_OK)

## 6 · Restore test data and measure dataset statistics
This restores each evaluated dataset from its verified Drive cache. That is needed for the baseline, speed and confusion-matrix steps. It then **measures**, per split actually used: image resolution, labelled instances per image, and box-size distribution in native pixels, using the COCO size bins small < 32², medium < 96², large ≥ 96². For E2 the train statistics use the exact 20,000-image subset list saved by notebook 02.

In [ ]:
# 6. Restore + dataset statistics
ROOTS, DSTATS = {}, []
for e in EVAL:
    ROOTS[e["id"]] = restore_dataset(e["variant"], e["gate"]["fingerprint"])
    if not CFG["measure_dataset_stats"]:
        continue
    cache = DIRS["cache"] / f"dstats_{e['id']}_{e['gate']['fingerprint']}.json"
    st = read_json(cache)
    if st is None:
        st = []
        subset = e["rec"].get("train_subset")
        keep_train = set(Path(subset["list_file"]).read_text().split()) if subset and Path(subset["list_file"]).exists() else None
        for split in ("train", "val", "test"):
            imgs = split_files(ROOTS[e["id"]], split)
            if split == "train" and keep_train is not None:
                imgs = [p for p in imgs if p.name in keep_train]
            res, inst, areas = Counter(), [], []
            for p in imgs:
                with Image.open(p) as im:
                    w, h = im.size
                res[f"{w}x{h}"] += 1
                lines = [l for l in (ROOTS[e["id"]] / "labels" / split / f"{p.stem}.txt").read_text().splitlines() if l.strip()]
                inst.append(len(lines))
                for ln in lines:
                    _, _, _, bw, bh = map(float, ln.split())
                    areas.append(bw * w * bh * h)
            a = np.array(areas) if areas else np.zeros(1)
            st.append({"experiment": e["id"], "split": split, "images": len(imgs),
                       "resolution": ", ".join(f"{k} ({v})" for k, v in res.most_common(3)),
                       "instances": int(sum(inst)), "instances_per_image_mean": float(np.mean(inst)) if inst else 0.0,
                       "instances_per_image_median": float(np.median(inst)) if inst else 0.0,
                       "box_area_median_px": float(np.median(a)),
                       "small_pct": float((a < 32**2).mean() * 100), "medium_pct": float(((a >= 32**2) & (a < 96**2)).mean() * 100),
                       "large_pct": float((a >= 96**2).mean() * 100)})
        write_json(cache, st)
    DSTATS += st
DSTATS_DF = pd.DataFrame(DSTATS)
if len(DSTATS_DF):
    print(DSTATS_DF.round(2).to_string(index=False))

## 7 · Pretrained baseline (zero-shot, before fine-tuning) 🔍
The COCO-pretrained `yolov8n.pt` (the **same file** used as the fine-tuning initialisation; its sha256 is checked against notebook 02's preflight) is evaluated on each **test** split. The evaluation protocol is identical to the fine-tuned models: `model.val()`, image size 640, conf 0.001, NMS IoU 0.7.

**Class mapping.** The model's classification head is restricted to the mapped COCO classes, by slicing those rows out of the head. Without this, the baseline would be charged for persons, signs and other irrelevant classes. Slicing was verified to leave the kept classes' boxes and scores unchanged.
- **UA-DETRAC:** COCO car + bus + truck are merged into `vehicle`.
- **BDD100K:** car→car, bus→bus, truck→truck, motorcycle→motorcycle.

**Caveat.** COCO's class definitions differ from each dataset's. For example, BDD `truck` here includes trailers, and UA-DETRAC `vehicle` includes vans and "others". So the baseline measures how far the off-the-shelf detector is from the dataset's labelling convention, as well as its visual ability.

In [ ]:
# 7. Pretrained baseline
from ultralytics import YOLO

PRETRAINED = DIRS["checkpoints"] / "pretrained" / "yolov8n.pt"
BASELINE = {}


def sliced_coco_model(keep, names, out):
    m = YOLO(str(PRETRAINED))
    det = m.model.model[-1]
    idx = torch.tensor(keep)
    for seq in det.cv3:
        conv = seq[-1]
        new = torch.nn.Conv2d(conv.in_channels, len(keep), 1)
        new.weight.data = conv.weight.data[idx].clone()
        new.bias.data = conv.bias.data[idx].clone()
        seq[-1] = new
    det.nc, det.no = len(keep), len(keep) + det.reg_max * 4
    m.model.nc, m.model.names = len(keep), {i: n for i, n in enumerate(names)}
    if hasattr(m.model, "yaml"):
        m.model.yaml["nc"] = len(keep)
    m.save(str(out))
    return out


if CFG["run_baseline"]:
    assert PRETRAINED.exists(), f"{PRETRAINED} missing - it is downloaded by notebook 02."
    sha = file_hash(PRETRAINED)
    assert PRETRAINED_SHA is None or sha == PRETRAINED_SHA, "pretrained weights differ from those used for fine-tuning"
    for e in EVAL:
        mp = CFG["baseline_mapping"].get(e["variant"])
        if mp is None:
            print(f"{e['id']}: no baseline mapping configured - baseline not available.")
            continue
        key = stable_hash({"sha": sha, "map": mp, "eval": CFG["eval"], "test": e["gate"]["manifest"]["test"]})
        cache = DIRS["cache"] / f"baseline_{e['variant']}_{key}.json"   # shared by experiments on the same test split
        res = read_json(cache)
        if res is None:
            log(f"{e['id']}: evaluating the zero-shot COCO baseline on the test split ...")
            mfile = sliced_coco_model(mp["coco_ids"], mp["coco_names"], DIRS["local_tmp"] / f"coco_mapped_{e['id']}.pt")
            names = e["rec"]["classes"]
            m = YOLO(str(mfile)).val(data=str(eval_yaml(ROOTS[e["id"]], names, e["id"])), split="test",
                                     imgsz=CFG["eval"]["imgsz"], batch=CFG["eval"]["batch"], conf=CFG["eval"]["conf"],
                                     iou=CFG["eval"]["iou"], single_cls=mp["single_cls"], device=DEVICE, plots=False,
                                     project=str(DIRS["local_tmp"]), name=f"baseline_{e['id']}", exist_ok=True,
                                     verbose=False)
            P, R, m50, m5095 = (float(x) for x in m.mean_results())
            per_class = []
            for row in m.summary():
                cname = names[0] if mp["single_cls"] else row["Class"]
                per_class.append({"class": cname, "instances": int(row["Instances"]), "precision": float(row["Box-P"]),
                                  "recall": float(row["Box-R"]), "f1": float(row["Box-F1"]),
                                  "mAP50": float(row["mAP50"]), "mAP50_95": float(row["mAP50-95"])})
            res = {"experiment": e["id"], "model": "yolov8n.pt (COCO-pretrained, no fine-tuning)",
                   "pretrained_sha256": sha, "mapping": mp, "split": "test", "device": ENV3["gpu"] or "cpu",
                   "overall": {"precision": P, "recall": R, "f1": f1(P, R), "mAP50": m50, "mAP50_95": m5095},
                   "per_class": per_class}
            write_json(cache, res)
        BASELINE[e["id"]] = {**res, "experiment": e["id"]}
        b, t = res["overall"], e["rec"]["test"]["overall"]
        print(f"{e['id']} test  baseline mAP50={b['mAP50']:.3f} mAP50-95={b['mAP50_95']:.3f}  |  "
              f"fine-tuned mAP50={t['mAP50']:.3f} mAP50-95={t['mAP50_95']:.3f}")
else:
    print("Baseline disabled (run_baseline=False) - reported as not available.")

## 8 · Same-device inference speed
Every fine-tuned `best.pt` is timed on **this** runtime with one protocol:
- `predict()`, batch 1, FP32, image size 640;
- the first 10 images are warm-up, then up to 200 test images are timed;
- each `Results.speed` is averaged over pre-processing, inference and post-processing.

FPS = 1000 / (mean end-to-end ms). It is a PyTorch measurement on the Colab device named in the table, **not** a TensorRT or Jetson measurement. The original per-run timings from notebook 02 are kept in the table, with their own device.

In [ ]:
# 8. Speed (same device for all)
SPEED = {}
if CFG["remeasure_speed"]:
    s = CFG["speed"]
    for e in EVAL:
        dev = ENV3["gpu"] or f"cpu ({os.cpu_count()} vCPU)"
        key = stable_hash({"w": file_hash(e["best"]), "dev": dev, "s": s})
        cache = DIRS["cache"] / f"speed_{e['id']}_{key}.json"
        res = read_json(cache)
        if res is None:
            log(f"{e['id']}: timing best.pt on {dev} ...")
            paths = [str(p) for p in split_files(ROOTS[e["id"]], "test")[: s["images"] + s["warmup"]]]
            rows = []
            for i, r in enumerate(YOLO(str(e["best"])).predict(source=paths, imgsz=s["imgsz"], batch=1, device=DEVICE,
                                                                conf=0.25, verbose=False, stream=True)):
                if i >= s["warmup"]:
                    rows.append(r.speed)
            mean = {k: float(np.mean([r[k] for r in rows])) for k in rows[0]}
            tot = sum(mean.values())
            res = {"device": dev, "images_timed": len(rows), "batch": 1, "precision": "fp32", "imgsz": s["imgsz"],
                   **{f"{k}_ms": v for k, v in mean.items()}, "total_ms": tot, "fps_end_to_end": 1000 / tot}
            write_json(cache, res)
        SPEED[e["id"]] = res
        print(f"{e['id']}: {res['total_ms']:.2f} ms/img end-to-end = {res['fps_end_to_end']:.1f} FPS on {res['device']}")
else:
    print("Speed re-measurement disabled - only the original per-run timings are reported.")

## 9 · Confusion matrices at confidence 0.25 (visual only)
These are generated on the test split with the fine-tuned `best.pt`, purely to visualise which classes get confused with each other. **No metric from this pass is reported, and nothing is selected from it.** The reported P/R/F1/mAP remain those of the standard evaluation.

In [ ]:
# 9. Confusion matrices (visual)
CM_FILES = {}
if CFG["confusion_matrix_conf"]:
    for e in EVAL:
        out = e["run_dir"] / f"cm_test_conf{CFG['confusion_matrix_conf']}"
        if not (out / "confusion_matrix_normalized.png").exists():
            log(f"{e['id']}: confusion matrix at conf {CFG['confusion_matrix_conf']} ...")
            YOLO(str(e["best"])).val(data=str(eval_yaml(ROOTS[e["id"]], e["rec"]["classes"], e["id"])), split="test",
                                     imgsz=CFG["eval"]["imgsz"], batch=CFG["eval"]["batch"],
                                     conf=CFG["confusion_matrix_conf"], iou=CFG["eval"]["iou"], device=DEVICE,
                                     plots=True, project=str(e["run_dir"]), name=out.name, exist_ok=True, verbose=False)
        for f in ("confusion_matrix.png", "confusion_matrix_normalized.png"):
            if (out / f).exists():
                dst = DIRS["final"] / f"{e['id']}_test_conf{CFG['confusion_matrix_conf']}_{f}"
                shutil.copy2(out / f, dst)
                CM_FILES.setdefault(e["id"], []).append(dst.name)
print(CM_FILES)

## 10 · Comparison table 🔍
There is one row per planned experiment, with every requested field. Metrics are on the **held-out test split** unless a column name says otherwise. Missing or unsuitable experiments keep **empty** metric cells, and their reason is given. The table is exported as `metrics/final_comparison.csv` and `metrics/final_comparison.xlsx`.

In [ ]:
# 10. Comparison table
from IPython.display import display

ROWS = []
for e in EXPS:
    row = {"experiment": e["id"],
           "dataset": DATASET_NOTES.get(e["dataset"], {}).get("name", DATASET_NAMES.get(e["dataset"], e["dataset"])),
           "status": e["status"], "reason_if_not_evaluated": e["reason"] if e["status"] != "EVALUATED" else ""}
    if e["status"] == "EVALUATED":
        rec, tr, te, va = e["rec"], e["rec"]["training"], e["rec"]["test"]["overall"], e["rec"]["validation"]["overall"]
        dc = rec["data_counts"]
        sub = rec.get("train_subset")
        b = BASELINE.get(e["id"], {}).get("overall", {})
        sp = SPEED.get(e["id"], {})
        row.update({
            "dataset_version": dataset_version(e),
            "data_gate_fingerprint": e["gate"].get("fingerprint"),
            "train_images": dc["train"]["images"],
            "train_images_note": f"fixed subset of {sub['n']} / {sub['of']} (seed {sub['seed']})" if sub else "full split",
            "val_images": dc["val"]["images"], "test_images": dc["test"]["images"],
            "num_classes": len(rec["classes"]), "classes": ", ".join(rec["classes"]),
            "train_instances": dc["train"]["instances_total"], "val_instances": dc["val"]["instances_total"],
            "test_instances": dc["test"]["instances_total"],
            "model": "YOLOv8n", "parameters": rec["model_stats"]["parameters"],
            "gflops_640": rec["model_stats"]["gflops_at_imgsz"],
            "initialisation": "COCO-pretrained yolov8n.pt (Ultralytics), fine-tuned",
            "epochs_completed": tr["epochs_completed"], "epochs_planned": tr["epochs_planned"],
            "best_epoch": tr["best_epoch"],
            "stop_reason": tr.get("stop_reason") or ("completed full epoch budget"
                                                     if tr["epochs_completed"] >= tr["epochs_planned"]
                                                     else "early stopping (validation patience)"),
            "precision": te["precision"], "recall": te["recall"], "f1": te["f1"],
            "mAP50": te["mAP50"], "mAP50_95": te["mAP50_95"], "val_mAP50_95": va["mAP50_95"],
            "baseline_mAP50": b.get("mAP50"), "baseline_mAP50_95": b.get("mAP50_95"),
            "gain_mAP50_vs_baseline": (te["mAP50"] - b["mAP50"]) if b else None,
            "gain_mAP50_95_vs_baseline": (te["mAP50_95"] - b["mAP50_95"]) if b else None,
            "training_time_h": (tr.get("training_time_sum_epochs_s") or 0) / 3600,
            "training_device": ", ".join(sorted({s.get("gpu") or "cpu" for s in e["status_json"].get("sessions", [])}))
                               or "unrecorded",
            "latency_ms_same_device": sp.get("total_ms"), "fps_same_device": sp.get("fps_end_to_end"),
            "speed_device_same": sp.get("device"),
            "latency_ms_original_run": rec["speed"].get("total_ms"), "fps_original_run": rec["speed"].get("fps_end_to_end"),
            "speed_device_original_run": rec["speed"].get("device"),
            "checkpoint_mb": rec["model_stats"]["weights_file_mb"], "best_checkpoint": str(e["best"]),
            "held_out_verified": HELD_OUT_OK.get(e["id"]),
        })
    ROWS.append(row)
COMP = pd.DataFrame(ROWS)
COMP.to_csv(DIRS["metrics"] / "final_comparison.csv", index=False)

PER_CLASS = []
for e in EVAL:
    bmap = {r["class"]: r for r in BASELINE.get(e["id"], {}).get("per_class", [])}
    train_inst = e["rec"]["data_counts"]["train"]["instances"]
    for r in e["rec"]["test"]["per_class"]:
        poor = (r["mAP50"] is not None and r["mAP50"] < CFG["poor_class_map50"]) or \
               (r["recall"] is not None and r["recall"] < CFG["poor_class_recall"])
        PER_CLASS.append({"experiment": e["id"], "dataset": DATASET_NOTES[e["dataset"]]["name"], "class": r["class"],
                          "train_instances": train_inst.get(r["class"]), "test_instances": r["instances"],
                          "precision": r["precision"], "recall": r["recall"], "f1": r["f1"], "mAP50": r["mAP50"],
                          "mAP50_95": r["mAP50_95"], "baseline_mAP50": bmap.get(r["class"], {}).get("mAP50"),
                          "baseline_mAP50_95": bmap.get(r["class"], {}).get("mAP50_95"),
                          "poor": bool(poor) if r["mAP50"] is not None else None})
PER_CLASS_DF = pd.DataFrame(PER_CLASS)
PER_CLASS_DF.to_csv(DIRS["metrics"] / "final_per_class.csv", index=False)
print("CSV written:", DIRS["metrics"] / "final_comparison.csv", "|", DIRS["metrics"] / "final_per_class.csv",
      "(the Excel workbook is written in cell 10b, with every sheet)")
key_cols = ["experiment", "dataset", "status", "num_classes", "train_images", "test_images", "precision", "recall",
            "f1", "mAP50", "mAP50_95", "baseline_mAP50", "epochs_completed", "training_time_h", "fps_same_device"]
display(COMP[[c for c in key_cols if c in COMP]].round(3))

## 10b · Results reported by the paper (quoted, not reproduced) 🔍
These values are **copied verbatim** from Raza et al. (2025): Tables 12, 13 and 15. They are not produced by these notebooks. They appear in their own table and are never mixed into our metric columns, because:
* the paper's dataset is private, Karachi traffic with 8 South-Asian vehicle classes;
* the paper does not state the IoU threshold behind its "mAP", so it can't be matched to mAP@50 or mAP@50–95;
* the paper reports no precision, recall or F1;
* its FPS figures come from a Tesla T4 workstation (Table 12) and from Jetson devices with TensorRT (Table 15), with no stated timing protocol.

The second table puts our YOLOv8n rows next to the paper's YOLOv8n rows, with a column explaining why they are not directly comparable.

In [ ]:
# 10b. Paper-reported results (quoted)
names_map = {e["id"]: DATASET_NOTES[e["dataset"]]["name"] for e in EVAL}
PAPER_ROWS = []
for model, d in PAPER["table12"]["models"].items():
    for scale in ("RGB", "Gray", "Green"):
        PAPER_ROWS.append({"source": "Table 12", "model": model, "data / condition": f"own dataset, {scale} input",
                           "mAP_reported_pct": d["mAP"][scale], "fps_reported": d["fps"][scale],
                           "hardware": PAPER["table12"]["hardware"], "model_size_mb": d["size_mb"]})
for model, d in PAPER["table13"].items():
    for ds, (un, au) in d.items():
        for cond, v in (("unaugmented", un), ("augmented", au)):
            PAPER_ROWS.append({"source": "Table 13", "model": model, "data / condition": f"{ds}, {cond} training",
                               "mAP_reported_pct": v, "fps_reported": None, "hardware": None, "model_size_mb": None})
for site, d in PAPER["table15"].items():
    for model, (m, nano, xavier) in d.items():
        for dev, fps in (("Jetson Nano", nano), ("Jetson Xavier", xavier)):
            PAPER_ROWS.append({"source": "Table 15", "model": model, "data / condition": f"{site} (edge, TensorRT)",
                               "mAP_reported_pct": m, "fps_reported": fps, "hardware": dev, "model_size_mb": None})
PAPER_DF = pd.DataFrame(PAPER_ROWS)
PAPER_DF.to_csv(DIRS["metrics"] / "paper_reported_results.csv", index=False)

t12 = PAPER["table12"]["models"]["YOLOv8-nano"]
SIDE = []
for e in EVAL:
    t, sp = e["rec"]["test"]["overall"], SPEED.get(e["id"], {})
    tr = e["rec"]["training"]
    SIDE.append({"source": f"this work · {e['id']}", "dataset": names_map[e["id"]],
                 "classes": len(e["rec"]["classes"]), "train_images": e["rec"]["data_counts"]["train"]["images"],
                 "epochs": f"{tr['epochs_completed']}/{tr['epochs_planned']}",
                 "map_value": f"mAP@50 {t['mAP50']:.3f} / mAP@50-95 {t['mAP50_95']:.3f}",
                 "fps": f"{sp['fps_end_to_end']:.1f} ({sp['device']}, PyTorch FP32)" if sp else "n/a",
                 "model_size_mb": f"{e['rec']['model_stats']['weights_file_mb']:.1f}",
                 "comparable": "reference (our protocol)"})
SIDE.append({"source": "paper · Table 12 (RGB)", "dataset": "private Karachi dataset", "classes": 8,
             "train_images": "2,182 (augmented)", "epochs": "100",
             "map_value": f"mAP {t12['mAP']['RGB']:.2f}% (IoU threshold not stated)",
             "fps": f"{t12['fps']['RGB']} (Tesla T4, protocol not stated)", "model_size_mb": f"{t12['size_mb']:.1f}",
             "comparable": "NO - different dataset/classes; mAP definition and FPS protocol unknown"})
for site, d in PAPER["table15"].items():
    m, _, xav = d["YOLOv8-nano"]
    SIDE.append({"source": f"paper · Table 15 ({site})", "dataset": f"{site} intersection (private)", "classes": 8,
                 "train_images": "2,182 (augmented)", "epochs": "100", "map_value": f"mAP {m}% (IoU threshold not stated)",
                 "fps": f"{xav:g} (Jetson Xavier NX, TensorRT)", "model_size_mb": "not reported",
                 "comparable": "NO - different dataset/classes and device (edge, TensorRT)"})
SIDE_DF = pd.DataFrame(SIDE)
SIDE_DF.to_csv(DIRS["metrics"] / "yolov8n_this_work_vs_paper.csv", index=False)
try:
    with pd.ExcelWriter(DIRS["metrics"] / "final_comparison.xlsx") as xw:
        COMP.to_excel(xw, sheet_name="comparison", index=False)
        PER_CLASS_DF.to_excel(xw, sheet_name="per_class_test", index=False)
        if len(DSTATS_DF):
            DSTATS_DF.to_excel(xw, sheet_name="dataset_stats", index=False)
        VERIFY_DF.to_excel(xw, sheet_name="held_out_checks", index=False)
        PAPER_DF.to_excel(xw, sheet_name="paper_reported (quoted)", index=False)
        SIDE_DF.to_excel(xw, sheet_name="yolov8n_vs_paper", index=False)
    print("Excel written:", DIRS["metrics"] / "final_comparison.xlsx")
except ImportError as err:
    print("Excel export skipped (no Excel engine installed):", err)
print("Reported by the paper (quoted; not reproduced):", PAPER["citation"])
display(PAPER_DF.pivot_table(index=["source", "data / condition"], columns="model", values="mAP_reported_pct",
                             aggfunc="first"))
print("\nYOLOv8n - this work vs. values reported by the paper:")
display(SIDE_DF)

## 11 · Plots
Four plots are saved to `reports/final_report/` and `plots/`:
1. Test detection metrics per experiment.
2. Pretrained baseline vs. fine-tuned.
3. Training curves: validation mAP and losses per epoch, from `epochs.csv`.
4. Per-class test mAP@50 and recall, with the poor-class thresholds drawn.

Colours follow the experiment (E1 blue, E2 orange) in every figure. The baseline is grey. All figures use one y-axis.

In [ ]:
# 11. Plots
import matplotlib.pyplot as plt

SURF, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
EXP_COLOR = {e["id"]: c for e, c in zip(EVAL, ["#2a78d6", "#eb6834", "#1baf7a"])}   # validated categorical slots
BASE_COLOR = "#a8a7a2"
plt.rcParams.update({"figure.facecolor": SURF, "axes.facecolor": SURF, "axes.edgecolor": GRID, "axes.labelcolor": INK2,
                     "xtick.color": INK2, "ytick.color": INK2, "text.color": INK, "axes.grid": True,
                     "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False})
FIGS = []


def save(fig, name):
    for d in (DIRS["final"], DIRS["plots"]):
        fig.savefig(d / name, dpi=150)
    FIGS.append(name)
    plt.show()
    plt.close(fig)


# 1) test metrics
mets = [("precision", "Precision"), ("recall", "Recall"), ("f1", "F1"), ("mAP50", "mAP@50"), ("mAP50_95", "mAP@50–95")]
fig, ax = plt.subplots(figsize=(8.5, 4), layout="constrained")
w = 0.8 / len(EVAL)
for i, e in enumerate(EVAL):
    t = e["rec"]["test"]["overall"]
    ax.bar(np.arange(len(mets)) + (i - (len(EVAL) - 1) / 2) * w, [t[k] for k, _ in mets], width=w * 0.92,
           color=EXP_COLOR[e["id"]], label=f"{e['id']} · {DATASET_NOTES[e['dataset']]['name']}")
ax.set_xticks(range(len(mets)), [m for _, m in mets]); ax.set_ylim(0, 1); ax.set_ylabel("score (test split)")
ax.set_title("Fine-tuned YOLOv8n — detection metrics on held-out test splits", loc="left")
ax.legend(loc="upper right")
save(fig, "final_1_test_metrics.png")

# 2) baseline vs fine-tuned
if BASELINE:
    fig, axes = plt.subplots(1, len(EVAL), figsize=(4.2 * len(EVAL), 3.8), sharey=True, squeeze=False, layout="constrained")
    for ax, e in zip(axes[0], EVAL):
        t, b = e["rec"]["test"]["overall"], BASELINE.get(e["id"], {}).get("overall")
        x = np.arange(2)
        if b:
            ax.bar(x - 0.2, [b["mAP50"], b["mAP50_95"]], 0.37, color=BASE_COLOR, label="pretrained (zero-shot)")
        ax.bar(x + 0.2, [t["mAP50"], t["mAP50_95"]], 0.37, color=EXP_COLOR[e["id"]], label="fine-tuned")
        ax.set_xticks(x, ["mAP@50", "mAP@50–95"]); ax.set_ylim(0, 1)
        ax.set_title(f"{e['id']} · {DATASET_NOTES[e['dataset']]['name']}", loc="left"); ax.legend(loc="upper right", fontsize=8)
    axes[0][0].set_ylabel("score (test split)")
    fig.suptitle("Effect of fine-tuning: COCO-pretrained baseline vs fine-tuned", x=0.01, ha="left")
    save(fig, "final_2_baseline_vs_finetuned.png")

# 3) training curves
fig, axes = plt.subplots(1, 4, figsize=(17, 3.9), layout="constrained")
for e in EVAL:
    ep, c = e["epochs"], EXP_COLOR[e["id"]]
    axes[0].plot(ep["epoch"], ep["metrics/mAP50-95(B)"], color=c, lw=2, label=f"{e['id']} val mAP@50–95")
    axes[0].plot(ep["epoch"], ep["metrics/mAP50(B)"], color=c, lw=2, ls=":", label=f"{e['id']} val mAP@50")
    for ax, comp in zip(axes[1:], ("box", "cls", "dfl")):
        ax.plot(ep["epoch"], ep[f"train/{comp}_loss"], color=c, lw=2, label=f"{e['id']} train")
        ax.plot(ep["epoch"], ep[f"val/{comp}_loss"], color=c, lw=2, ls="--", label=f"{e['id']} val")
axes[0].set_title("validation mAP", loc="left"); axes[0].set_ylim(0, 1)
for ax, comp in zip(axes[1:], ("box", "cls", "dfl")):
    ax.set_title(f"{comp} loss", loc="left")
for ax in axes:
    ax.set_xlabel("epoch"); ax.legend(fontsize=7)
fig.suptitle("Training curves (from results.csv) — solid: train / mAP@50–95, dashed: val loss, dotted: val mAP@50",
             x=0.01, ha="left")
save(fig, "final_3_training_curves.png")

# 4) per-class
if len(PER_CLASS_DF):
    pc = PER_CLASS_DF.dropna(subset=["mAP50"]).reset_index(drop=True)
    labels = [f"{r.experiment} · {r['class']}" for _, r in pc.iterrows()]
    fig, axes = plt.subplots(1, 2, figsize=(11, 0.45 * len(pc) + 1.6), sharey=True, layout="constrained")
    for ax, (col, thr, title) in zip(axes, (("mAP50", CFG["poor_class_map50"], "test mAP@50"),
                                            ("recall", CFG["poor_class_recall"], "test recall"))):
        ax.barh(range(len(pc)), pc[col], color=[EXP_COLOR[x] for x in pc["experiment"]], height=0.7)
        ax.axvline(thr, color=INK2, lw=1, ls="--")
        ax.set_xlim(0, 1); ax.set_title(f"{title}  (dashed line: poor < {thr})", loc="left")
    axes[0].set_yticks(range(len(pc)), labels); axes[0].invert_yaxis()
    fig.suptitle("Per-class performance on held-out test splits", x=0.01, ha="left")
    save(fig, "final_4_per_class.png")
print("Figures:", FIGS)

## 12 · Per-class analysis 🔍
A class is flagged **poor** when its test mAP@50 < 0.50 **or** its test recall < 0.50. Both thresholds are configurable, and they are stated in the report. Training-instance counts are shown alongside, because data scarcity is the most common cause.

In [ ]:
# 12. Per-class analysis
show = PER_CLASS_DF[["experiment", "dataset", "class", "train_instances", "test_instances", "precision", "recall",
                     "f1", "mAP50", "mAP50_95", "baseline_mAP50", "poor"]]
display(show.round(3))
POOR = PER_CLASS_DF[PER_CLASS_DF["poor"] == True]
print("Poor classes:", [f"{r.experiment}:{r['class']}" for _, r in POOR.iterrows()] or "none")

## 13 · Markdown report
This writes `reports/final_report/final_report.md` next to the figures. **Every number in it comes from the tables above.** The discussion is generated from those values plus the recorded run metadata (stop reasons, subsets, devices, label versions), so it cannot drift from the data.

In [ ]:
# 13. Markdown report
L = []
add = L.append
names = {e["id"]: DATASET_NOTES[e["dataset"]]["name"] for e in EVAL}
add("# SmartFlow — YOLOv8n vehicle detection: experiment comparison\n")
add(f"_Generated {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')} by notebook 03 from the saved run "
    "artefacts. No value in this report was entered by hand._\n")
add("## 1. Scope and claim boundaries\n")
add("- **Model:** YOLOv8n, the vehicle-detection component of Raza et al., *IEEE Access* 13 (2025), "
    "doi:10.1109/ACCESS.2025.3602844.")
add("- **Fine-tuning:** from COCO-pretrained weights with the paper's reported YOLOv8n training settings "
    "(AdamW, lr0 0.001, batch 16).")
add("- **Not a reproduction of the paper.** The paper's Karachi dataset (8 South-Asian vehicle classes) is not "
    "public, and the paper does not report its mAP IoU threshold, input size or FPS protocol. The paper also compares "
    "YOLOv7-tiny, YOLOv8-s and YOLO-NAS-s, which were not run here.")
add("- **Metric terminology:** mAP (mean average precision) is a **detection** metric that combines localisation "
    "(IoU) and ranking quality. It is not classification accuracy, and is reported as mAP@50 and mAP@50–95.\n")

add("## 2. Experiments and status\n")
add(md_table([{"experiment": e["id"],
               "dataset": DATASET_NOTES.get(e["dataset"], {}).get("name", DATASET_NAMES.get(e["dataset"], e["dataset"])),
               "status": e["status"], "reason": (e["reason"] or "—").replace("|", "/")} for e in EXPS],
             ["experiment", "dataset", "status", "reason"]))
add("\nExperiments that were not evaluated have **no metric values**. Their reason is given above.\n")

add("## 3. Comparison on held-out test splits\n")
add(md_table(ROWS, ["experiment", "dataset", "num_classes", "train_images", "val_images", "test_images",
                    "test_instances", "precision", "recall", "f1", "mAP50", "mAP50_95"],
             ["Exp.", "Dataset", "Classes", "Train img", "Val img", "Test img", "Test inst.", "P", "R", "F1",
              "mAP@50", "mAP@50–95"]))
add("\n" + md_table(ROWS, ["experiment", "dataset_version", "train_images_note", "epochs_completed", "epochs_planned", "best_epoch",
                           "stop_reason", "training_time_h", "checkpoint_mb", "parameters", "gflops_640"],
                    ["Exp.", "Dataset version", "Training data", "Epochs done", "Planned", "Best epoch", "Stop reason",
                     "Train time (h)", "Checkpoint (MB)", "Params", "GFLOPs@640"]))
add("\nFull table: `metrics/final_comparison.csv` / `.xlsx`.\n")
add("![test metrics](final_1_test_metrics.png)\n")

add("## 4. Pretrained baseline vs. fine-tuned\n")
if BASELINE:
    brows = [{"experiment": e["id"], "dataset": names[e["id"]],
              "b50": BASELINE[e["id"]]["overall"]["mAP50"], "b5095": BASELINE[e["id"]]["overall"]["mAP50_95"],
              "f50": e["rec"]["test"]["overall"]["mAP50"], "f5095": e["rec"]["test"]["overall"]["mAP50_95"],
              "d50": e["rec"]["test"]["overall"]["mAP50"] - BASELINE[e["id"]]["overall"]["mAP50"]}
             for e in EVAL if e["id"] in BASELINE]
    add(md_table(brows, ["experiment", "dataset", "b50", "b5095", "f50", "f5095", "d50"],
                 ["Exp.", "Dataset", "Baseline mAP@50", "Baseline mAP@50–95", "Fine-tuned mAP@50",
                  "Fine-tuned mAP@50–95", "Δ mAP@50"]))
    add("\n**How the baseline was measured.** The baseline is the **same** COCO-pretrained `yolov8n.pt` used to "
        "initialise fine-tuning, evaluated zero-shot on the same test split with the same protocol. Its head was "
        "restricted to the mapped COCO classes:")
    for e in EVAL:
        if e["id"] in BASELINE:
            mp = BASELINE[e["id"]]["mapping"]
            add(f"- {e['id']}: COCO {', '.join(mp['coco_names'])} → {', '.join(e['rec']['classes'])}"
                + (" (merged into one class)" if mp["single_cls"] else ""))
    add("\nBecause COCO and each dataset define classes differently, part of the gap reflects **labelling "
        "conventions**, not only visual ability.\n")
    add("![baseline vs fine-tuned](final_2_baseline_vs_finetuned.png)\n")
else:
    add("Baseline not measured (disabled). **No baseline values are reported.**\n")

add("## 5. Per-class performance\n")
add(f"A class is flagged **poor** if its test mAP@50 < {CFG['poor_class_map50']} or its recall < "
    f"{CFG['poor_class_recall']}.\n")
add(md_table(PER_CLASS, ["experiment", "class", "train_instances", "test_instances", "precision", "recall", "f1",
                         "mAP50", "mAP50_95", "poor"],
             ["Exp.", "Class", "Train inst.", "Test inst.", "P", "R", "F1", "mAP@50", "mAP@50–95", "Poor"]))
if len(POOR):
    add("\n**Poor classes:** " + ", ".join(f"{r.experiment} `{r['class']}` (mAP@50 {r.mAP50:.3f}, recall "
                                            f"{r.recall:.3f}, {int(r.train_instances)} training instances)"
                                            for _, r in POOR.iterrows()) + ".")
add("\n![per class](final_4_per_class.png)\n")
for e in EVAL:
    for f in CM_FILES.get(e["id"], []):
        if "normalized" in f:
            add(f"![{e['id']} confusion matrix (conf {CFG['confusion_matrix_conf']}, visual only)]({f})\n")

add("## 6. Dataset differences\n")
add(md_table([{"exp": e["id"], **{k: DATASET_NOTES[e["dataset"]][k] for k in ("name", "domain", "class_definition",
                                                                              "annotation", "preparation")}}
              for e in EVAL], ["exp", "name", "domain", "class_definition", "annotation", "preparation"],
             ["Exp.", "Dataset", "Domain (documented)", "Classes", "Annotation (documented)", "Preparation"]))
if len(DSTATS_DF):
    add("\n**Measured from the label files (splits as used):**\n")
    add(md_table(DSTATS, ["experiment", "split", "images", "resolution", "instances_per_image_mean", "box_area_median_px",
                          "small_pct", "medium_pct", "large_pct"],
                 ["Exp.", "Split", "Images", "Resolution (top)", "Inst./img", "Median box area (px²)", "Small %",
                  "Medium %", "Large %"]))
add("")

add("## 7. Comparability\n")
cav = []
ncls = {e["id"]: len(e["rec"]["classes"]) for e in EVAL}
if len(set(ncls.values())) > 1:
    cav.append("**Different class definitions:** " + ", ".join(f"{k} has {v} class(es)" for k, v in ncls.items())
               + ". A 1-class task is easier, so mAP across these rows is **not directly comparable**.")
cav.append("**Different domains and annotation conventions:** " + "; ".join(
    f"{names[e['id']]}: {DATASET_NOTES[e['dataset']]['domain']}" for e in EVAL)
    + ". Annotation processes and label versions differ (Section 6).")
for e in EVAL:
    tr, sub = e["rec"]["training"], e["rec"].get("train_subset")
    if tr.get("stopped_by_user_at_epoch") or tr["epochs_completed"] < tr["epochs_planned"]:
        cav.append(f"**{e['id']} trained {tr['epochs_completed']} of {tr['epochs_planned']} epochs** "
                   f"({tr.get('stop_reason')}). Its best epoch was {tr['best_epoch']}, so it may be under-trained.")
    if sub:
        cav.append(f"**{e['id']} used a fixed subset of {sub['n']} of {sub['of']} training images** (seed {sub['seed']}).")
devs = {e["id"]: e["rec"]["speed"].get("device") for e in EVAL}
if len(set(devs.values())) > 1:
    cav.append("**Original speed timings were taken on different devices** (" + ", ".join(f"{k}: {v}" for k, v in devs.items())
               + "). Only the same-device re-measurement (Section 9) is comparable across experiments.")
cav.append("**Same evaluation protocol for all fine-tuned rows:** Ultralytics `val()`, image size 640, conf 0.001, "
           "NMS IoU 0.7, COCO-style AP over 10 IoU thresholds. Its confidence for P/R/F1 is the one that maximises mean F1.")
gaps = [(e["id"], e["rec"]["validation"]["overall"]["mAP50_95"], e["rec"]["test"]["overall"]["mAP50_95"]) for e in EVAL]
for i, v, t in gaps:
    if v - t > 0.05:
        cav.append(f"**{i}: validation mAP@50–95 ({v:.3f}) exceeds test ({t:.3f}) by {v - t:.3f}.** The validation "
                   "split is easier than or more similar to training than the test split.")
cav.append("**Not comparable with the paper's 90% mAP / 74 FPS:** the dataset, classes, mAP definition and FPS "
           "protocol all differ or are unreported.")
for c in cav:
    add(f"- {c}")
add("")

add("## 8. Held-out test verification\n")
add(md_table(VERIFY, ["experiment", "evidence", "result"], ["Exp.", "Evidence", "Result"]))
add("\nAll reported test metrics were computed **once**, after training, on the held-out test split with the "
    "validation-selected `best.pt`. Nothing was selected or tuned on test data.\n")

add("## 9. Efficiency\n")
add(md_table(ROWS, ["experiment", "latency_ms_same_device", "fps_same_device", "speed_device_same",
                    "latency_ms_original_run", "fps_original_run", "speed_device_original_run", "checkpoint_mb"],
             ["Exp.", "Latency (ms, same device)", "FPS (same device)", "Device", "Latency (ms, original)",
              "FPS (original)", "Device (original)", "Checkpoint (MB)"]))
add(f"\n**Method.** `predict()`, batch 1, FP32, image size {CFG['speed']['imgsz']}, {CFG['speed']['warmup']} "
    f"warm-up images, then up to {CFG['speed']['images']} timed test images. Latency is the mean of pre-processing, "
    "inference and post-processing time, and FPS = 1000 / latency. This is PyTorch on Colab hardware, **not** "
    "TensorRT or Jetson.\n")

add("## 10. Reproducibility\n")
e0 = EVAL[0]["rec"]
add(f"- **Software:** Python {e0['environment']['python']}, torch {e0['environment']['torch']}, ultralytics "
    f"{e0['environment']['ultralytics']}, numpy {e0['environment']['numpy']}. Report generated with torch "
    f"{ENV3['torch']} on {ENV3['gpu'] or 'CPU'}.")
add(f"- **Model:** YOLOv8n ({e0['model_stats']['parameters']:,} parameters), initialised from `yolov8n.pt` "
    f"(sha256 `{PRETRAINED_SHA or 'not recorded'}`).")
tc = e0["training_config"]
add(f"- **Training config (all experiments):** imgsz {tc['imgsz']}, epochs {tc['epochs']}, batch {tc['batch']}, "
    f"optimizer {tc['optimizer']}, lr0 {tc['lr0']}, patience {tc['patience']}, AMP {tc['amp']}, deterministic "
    f"{tc['deterministic']}, close_mosaic {tc['close_mosaic']}, seed {e0['training_config']['requested_args']['seed']}. "
    "Augmentation: Ultralytics defaults.")
for e in EVAL:
    r = e["rec"]
    add(f"- **{e['id']}** ({names[e['id']]}): data-gate fingerprint `{r['data_gate_fingerprint']}`; splits "
        f"{r['data_counts']['train']['images']}/{r['data_counts']['val']['images']}/{r['data_counts']['test']['images']}"
        + (f"; training subset list `{r['train_subset']['list_file']}`" if r.get("train_subset") else "")
        + f"; trained on {r['training']['sessions']} session(s), effective batch {r['training']['effective_batch']}; "
        f"checkpoint `{e['best']}`.")
add("- **Split construction:** UA-DETRAC is split by sequence (frames sub-sampled after splitting). BDD100K uses the "
    "official val as test and a seeded 10% of official train as val. Leakage checks are in Section 8.")
add("- **Artefacts:** `metrics/summary_all_experiments.csv`, `metrics/final_comparison.csv/.xlsx`, "
    "`metrics/final_per_class.csv`, per-run `report/metrics.json`, `reports/data_gate.json`.\n")

add("## 11. Results reported by the paper (quoted, not reproduced)\n")
add(f"Source: {PAPER['citation']}. Dataset: {PAPER['dataset']}. Training: {PAPER['training']}. "
    "The paper does not state the IoU threshold behind its mAP, and reports no precision, recall or F1.\n")
p12 = PAPER["table12"]["models"]
add(md_table([{"model": m, "size": f"{d['size_mb']:.1f}", "rgb": f"{d['mAP']['RGB']:.2f}",
               "gray": f"{d['mAP']['Gray']:.2f}", "green": f"{d['mAP']['Green']:.2f}",
               "fps": f"{d['fps']['RGB']:.2f}"} for m, d in p12.items()],
             ["model", "size", "rgb", "gray", "green", "fps"],
             ["Model (Table 12)", "Size (MB)", "mAP % RGB", "mAP % Gray", "mAP % Green", "FPS RGB (Tesla T4)"]))
add("\n" + md_table([{"site": site, "model": m, "map": f"{v[0]:g}", "nano": f"{v[1]:g}" if v[1] is not None else "–",
                      "xavier": f"{v[2]:g}"} for site, d in PAPER["table15"].items() for m, v in d.items()],
                    ["site", "model", "map", "nano", "xavier"],
                    ["Site (Table 15)", "Model", "mAP %", "Jetson Nano FPS", "Jetson Xavier FPS"]))
add("\n**YOLOv8n: this work next to the paper's reported values**\n")
add(md_table(SIDE, ["source", "dataset", "classes", "train_images", "epochs", "map_value", "fps", "model_size_mb",
                    "comparable"],
             ["Source", "Dataset", "Classes", "Train images", "Epochs", "mAP", "FPS (device)", "Size (MB)",
              "Directly comparable?"]))
add("\nThe checkpoint size matches the paper's YOLOv8n (about 6 MB), as expected for the same architecture. "
    "Accuracy and speed **cannot** be compared directly: the datasets, class sets, training length, mAP "
    "definition and timing hardware/protocol all differ. Note also that the paper's headline 74 FPS is its "
    "Tesla T4 figure (Table 12); on a Jetson Xavier NX it reports 33.5-35 FPS (Table 15).\n")

add("## 12. Discussion\n")
best = max(EVAL, key=lambda e: e["rec"]["test"]["overall"]["mAP50_95"])
txt = (f"Fine-tuned YOLOv8n reached the highest test mAP@50–95 on **{names[best['id']]}** "
       f"({best['rec']['test']['overall']['mAP50_95']:.3f}; mAP@50 {best['rec']['test']['overall']['mAP50']:.3f}).")
others = [e for e in EVAL if e is not best]
for e in others:
    t = e["rec"]["test"]["overall"]
    txt += (f" On **{names[e['id']]}** it reached mAP@50–95 {t['mAP50_95']:.3f} (mAP@50 {t['mAP50']:.3f}, "
            f"recall {t['recall']:.3f}).")
add(txt)
add("\nThis ordering should **not** be read as one dataset being intrinsically easier than another. The experiments "
    "differ in number of classes, viewpoint, object size, training-set size and training length (Section 7). "
    "Each row measures the detector on its own task.")
if BASELINE:
    gains = [f"{e['id']} {e['rec']['test']['overall']['mAP50'] - BASELINE[e['id']]['overall']['mAP50']:+.3f}"
             for e in EVAL if e["id"] in BASELINE]
    add(f"\nCompared with the COCO-pretrained baseline, fine-tuning changed test mAP@50 by: {', '.join(gains)}. "
        "This includes adapting to each dataset's class definitions and viewpoint.")
for e in EVAL:
    src_id = e["rec"].get("continued_from")
    src = next((x for x in EVAL if x["id"] == src_id), None)
    if src:
        a, b = src["rec"]["test"]["overall"], e["rec"]["test"]["overall"]
        add(f"\n**Effect of training longer ({src_id} → {e['id']}).** With identical data and settings, continuing "
            f"from epoch {src['rec']['training']['epochs_completed']} to {e['rec']['training']['epochs_completed']} "
            f"changed test mAP@50 from {a['mAP50']:.3f} to {b['mAP50']:.3f} ({b['mAP50'] - a['mAP50']:+.3f}) and "
            f"mAP@50–95 from {a['mAP50_95']:.3f} to {b['mAP50_95']:.3f} ({b['mAP50_95'] - a['mAP50_95']:+.3f}). "
            f"Recall changed from {a['recall']:.3f} to {b['recall']:.3f}. This is a single run, so run-to-run "
            "variance is not measured.")
if len(POOR):
    add("\nPer class, the weakest results coincide with the rarest training classes ("
        + ", ".join(f"`{r['class']}`: {int(r.train_instances)} training instances" for _, r in POOR.iterrows())
        + "). Low recall dominates, meaning these objects are mostly missed rather than misclassified.")
add("\n**Limitations.**")
lim = ["Only YOLOv8n was evaluated, with one seed. Run-to-run variance is unmeasured, so small differences "
       "should not be over-interpreted."]
lim += [c.replace("**", "") for c in cav if "trained" in c or "subset" in c]
lim.append("Datasets without detection boxes (see Section 2) could not be evaluated without manual annotation.")
for x in lim:
    add(f"- {x}")
add("\n**Recommended next experiment.**")
rec_next = []
continued = {e["rec"].get("continued_from") for e in EVAL}
for e in EVAL:
    tr = e["rec"]["training"]
    if e["id"] in continued:
        continue
    short = tr["epochs_completed"] < tr["epochs_planned"]
    sub = e["rec"].get("train_subset")
    if short or sub:
        what = " and ".join(x for x in (
            f"all {tr['epochs_planned']} epochs (it stopped at {tr['epochs_completed']})" if short else "",
            f"the full training split ({sub['of']} images instead of the {sub['n']}-image subset)" if sub else "") if x)
        rec_next.append(f"Re-train {e['id']} on a GPU with {what}, so that it matches the budget of the other "
                        "experiments.")
if len(POOR):
    rec_next.append("Address the poor classes with more training instances or class-balanced sampling, and report "
                    "per-class results again.")
rec_next.append("Evaluate the paper's other detectors (YOLOv8-s, YOLOv7-tiny, YOLO-NAS-s) under the identical "
                "protocol, and time them with TensorRT on the target edge device, before comparing with the "
                "paper's 74 FPS.")
for i, x in enumerate(rec_next, 1):
    add(f"{i}. {x}")

report_md = "\n".join(L) + "\n"
(DIRS["final"] / "final_report.md").write_text(report_md)
(DIRS["reports"] / "final_report.md").write_text(report_md.replace("](final_", "](final_report/final_"))
print(report_md[:3000], "\n...\n")
print("Report:", DIRS["final"] / "final_report.md")

## 14 · Output files

In [ ]:
# 14. Output files
for p in sorted([DIRS["metrics"] / "final_comparison.csv", DIRS["metrics"] / "final_comparison.xlsx",
                 DIRS["metrics"] / "final_per_class.csv", DIRS["final"] / "final_report.md"]
                + [DIRS["final"] / f for f in FIGS]):
    print(("OK  " if p.exists() else "--  ") + str(p))